In [6]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [7]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [8]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [9]:
def get_joke(state:JokeState):
    prompt = f'create a joke based on this topic: {state["topic"]}'
    res = model.invoke(prompt).content
    return {"joke":res}

def explain_joke(state:JokeState):
    prompt = f'create a short explanation of this joke: {state["joke"]}'
    res = model.invoke(prompt).content
    return {"explanation": res}

In [11]:
graph = StateGraph(JokeState)
checkpointer = InMemorySaver()

graph.add_node("get_joke",get_joke)
graph.add_node("explain_joke",explain_joke)

graph.add_edge(START,"get_joke")
graph.add_edge("get_joke","explain_joke")
graph.add_edge("explain_joke",END)

workflow = graph.compile(checkpointer=checkpointer)

In [12]:
config = {'configurable':{"thread_id":1}}
initial_state = {"topic":"Pizza"}
final_state = workflow.invoke(initial_state,config=config)
print(final_state)

{'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'This joke is a play on words. "Crusty" has a double meaning: it refers to the crust of a pizza, but it also means being irritable or grumpy. The punchline is funny because it\'s a clever and unexpected connection between the setup (the pizza being in a bad mood) and the word "crusty", creating a humorous pun.'}


In [14]:
workflow.get_state(config)

StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'This joke is a play on words. "Crusty" has a double meaning: it refers to the crust of a pizza, but it also means being irritable or grumpy. The punchline is funny because it\'s a clever and unexpected connection between the setup (the pizza being in a bad mood) and the word "crusty", creating a humorous pun.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f186818-346d-6f82-8002-ce20683b2d4f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-23T10:30:23.655885+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f186818-1d26-653c-8001-daf7410835de'}}, tasks=(), interrupts=())

In [16]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'This joke is a play on words. "Crusty" has a double meaning: it refers to the crust of a pizza, but it also means being irritable or grumpy. The punchline is funny because it\'s a clever and unexpected connection between the setup (the pizza being in a bad mood) and the word "crusty", creating a humorous pun.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f186818-346d-6f82-8002-ce20683b2d4f'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-23T10:30:23.655885+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f186818-1d26-653c-8001-daf7410835de'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!'}, next=('explain_joke',),

Time Travel

In [17]:
# Time travel
workflow.get_state({"configurable":{"thread_id":1,"checkpoint_id":"1f186818-1d26-653c-8001-daf7410835de"}})

StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!'}, next=('explain_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f186818-1d26-653c-8001-daf7410835de'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-07-23T10:30:21.214825+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f186817-a816-653d-8000-645d2927baa4'}}, tasks=(PregelTask(id='96049bbc-2a4d-357d-ac1f-7dcee8fa1f5a', name='explain_joke', path=('__pregel_pull', 'explain_joke'), error=None, interrupts=(), state=None, result={'explanation': 'This joke is a play on words. "Crusty" has a double meaning: it refers to the crust of a pizza, but it also means being irritable or grumpy. The punchline is funny because it\'s a clever and unexpected connection between the setup (the pizza being in a bad mood) and the word "crusty", creating a humorous pun.'}),), interrupts=(

In [18]:
workflow.invoke(None,{"configurable":{"thread_id":1,"checkpoint_id":"1f186818-1d26-653c-8001-daf7410835de"}})

{'topic': 'Pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!',
 'explanation': 'This joke is a play on words. "Crusty" has a double meaning: it refers to the crust of a pizza, but it also means being irritable or grumpy. The joke relies on this pun to create a humorous connection between the pizza\'s physical characteristic (its crust) and its emotional state (being in a bad mood).'}

In [ ]:
list(workflow.get_state_history(config)) # when time traveling from a specific point, it will create a new branch from that check point and the new state snapshots will be appended into the history

[StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'This joke is a play on words. "Crusty" has a double meaning: it refers to the crust of a pizza, but it also means being irritable or grumpy. The joke relies on this pun to create a humorous connection between the pizza\'s physical characteristic (its crust) and its emotional state (being in a bad mood).'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18682f-5f1e-6823-8003-0d62c75c59a2'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-23T10:40:45.533763+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18682e-c22f-63c9-8002-c1636b21d066'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!'}, next=('explain_joke',), confi

Updating the state: changing topic from pizza to samosa

In [21]:
workflow.update_state({"configurable":{"thread_id":1,"checkpoint_id":"1f186817-a816-653d-8000-645d2927baa4","checkpoint_ns":""}},{"topic":"samosa"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f186846-5bd0-6f41-8001-9684238cdfaf'}}

In [22]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'topic': 'samosa'}, next=('get_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f186846-5bd0-6f41-8001-9684238cdfaf'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-23T10:51:02.588977+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f186817-a816-653d-8000-645d2927baa4'}}, tasks=(PregelTask(id='252ce859-0c2d-d951-816e-c1edeee08721', name='get_joke', path=('__pregel_pull', 'get_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty!', 'explanation': 'This joke is a play on words. "Crusty" has a double meaning: it refers to the crust of a pizza, but it also means being irritable or grumpy. The joke relies on this pun to create a humorous connection between the pizza\'s physical characteristic (its 

In [23]:
# 1f186846-5bd0-6f41-8001-9684238cdfaf
workflow.invoke(None,{"configurable":{"thread_id":1,"checkpoint_id":"1f186846-5bd0-6f41-8001-9684238cdfaf"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it was feeling a little "crunchy" on the outside and "empty" on the inside! (get it?)',
 'explanation': 'This joke is a play on words. A samosa, a type of fried or baked pastry, is typically crunchy on the outside and empty or hollow on the inside. The joke uses this physical characteristic to make a humorous comparison to emotional feelings. The punchline suggests that the samosa is seeking therapy because it\'s feeling emotionally fragile ("crunchy" like its exterior) and unfulfilled ("empty" like its interior), making a lighthearted connection between the food\'s physical properties and human emotions.'}